# MUnitQuest – Algorithm Challenge Tutorial
# Dynamic Tasks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUnitQuest/MUnitQuest_tutorials/blob/main/algorithm_challenge_tutorials/02_familiarisation_dynamic.ipynb)


This notebook walks through the full workflow for MUnitQuest participants working with **dynamic contraction** recordings:

1. **Download** the familiarisation dataset for the dynamic task
2. **Explore** a recording – understand the file structure and the structure of dynamic trials
3. **Decompose** the HD-EMG with CBSS (the reference algorithm bundled with `muniverse`)
4. **Export** your spike trains as a TSV ready for codabench upload

---

### Installation

```bash
python -m venv .venv && source .venv/bin/activate
pip install muniverse-emg
```

In [ ]:
import sys, os
if "google.colab" in sys.modules:
    os.system("pip install muniverse-emg")
    os.system("git clone --depth 1 https://github.com/MUnitQuest/MUnitQuest_tutorials.git")
    sys.path.insert(0, "MUnitQuest_tutorials/algorithm_challenge_tutorials")

In [ ]:
import json
from importlib.resources import files
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

from muniverse.algorithms.decomposition import decompose_cbss
from muniverse.utils.bids_routines import read_edf

## 1 – Download the dataset

The familiarisation dataset is hosted on DaRUS, the data repository of the University of Stuttgart under the [MUnitQuest dataverse](https://darus.uni-stuttgart.de/dataverse/munitquest-algorithm-challenge). Use `muniverse` to download it.

Each recording is stored in **BIDS-flat** format: an EDF file containing the HD-EMG signals accompanied by sidecar metadata files:

- `*_emg.edf` — HD-EMG signal (all channels) + Torque + JointAngle
- `*_channels.tsv` — channel names, types, and units
- `*_emg.json` — recording metadata (sampling frequency, task name, …)
- `*_events.tsv` — behavioural phase events (rest, contraction, …)

In [ ]:
DATASET_ROOT = Path("../data/Dynamic")

In [ ]:
from muniverse.datasets import load_dataset
load_dataset("munitquest-familiarisation-dynamic", output_dir=DATASET_ROOT)

In [ ]:
def load_recording(edf_path):
    """Load a BIDS-flat EDF recording and its sidecar files.

    Parameters
    ----------
    edf_path : str or Path
        Path to the *_emg.edf file.

    Returns
    -------
    dict with keys:
        emg        - (n_channels, n_samples) HD-EMG in mV
        effort     - (n_samples,) normalised effort (0-1, fraction of MVC)
        angle      - (n_samples,) joint angle in degrees
        fsamp      - int, sampling frequency
        n_rows     - int, electrode grid rows (always 10 for this dataset)
        n_cols     - int, electrode grid columns
        channels   - DataFrame from *_channels.tsv
        electrodes - DataFrame from *_electrodes.tsv (name, x, y columns)
        events     - DataFrame from *_events.tsv (empty if not found)
        task_name  - str, recording description
        edf_path   - Path object
    """
    edf_path = Path(edf_path)
    stem = edf_path.stem  # e.g. sub-syn001_task-isometric_emg

    # Read EDF: signals shape is (n_channels, n_samples)
    signals, sig_hdrs, hdr = read_edf(str(edf_path))

    # Read channels TSV
    channels_tsv = edf_path.with_name(stem.replace("_emg", "_channels") + ".tsv")
    channels_df = pd.read_csv(channels_tsv, sep="\t")

    # Read electrodes TSV
    electrodes_tsv = edf_path.with_name(stem.replace("_emg", "_electrodes") + ".tsv")
    electrodes_df = pd.read_csv(electrodes_tsv, sep="\t") if electrodes_tsv.exists() else pd.DataFrame()

    # Read JSON sidecar
    json_path = edf_path.with_suffix(".json")
    with open(json_path) as f:
        meta = json.load(f)
    fsamp = int(meta["SamplingFrequency"])
    task_name = meta.get("TaskName", "")

    # Read events TSV (optional)
    events_tsv = edf_path.with_name(stem.replace("_emg", "_events") + ".tsv")
    if events_tsv.exists():
        events_df = pd.read_csv(events_tsv, sep="\t")
    else:
        events_df = pd.DataFrame()

    # Split channels by type
    emg_mask    = channels_df["type"].str.upper() == "EMG"
    torque_mask = channels_df["name"].str.lower() == "torque"
    angle_mask  = channels_df["name"].str.lower() == "jointangle"

    n_emg = int(emg_mask.sum())
    emg = signals[:n_emg]          # (n_emg_channels, n_samples)  [mV]

    # Torque channel → effort (scale %MVC to 0–1)
    torque_idx = int(np.where(torque_mask)[0][0])
    effort = signals[torque_idx] / 100.0

    # Joint angle channel
    angle_idx = int(np.where(angle_mask)[0][0])
    angle = signals[angle_idx]

    # All recordings use a 10-row electrode grid; n_cols derived from channel count
    n_rows = 10
    n_cols = n_emg // n_rows

    return {
        "emg":        emg,
        "effort":     effort,
        "angle":      angle,
        "fsamp":      fsamp,
        "n_rows":     n_rows,
        "n_cols":     n_cols,
        "channels":   channels_df,
        "electrodes": electrodes_df,
        "events":     events_df,
        "task_name":  task_name,
        "edf_path":   edf_path,
    }


In [ ]:
edf_path = DATASET_ROOT / "recording-023_challenge-dynamic/recording-023_challenge-dynamic_emg.edf"

rec = load_recording(edf_path)
print(f"Muscle/task: {rec['task_name']}")
print(f"Electrode grid: {rec['n_rows']} rows × {rec['n_cols']} cols = {rec['emg'].shape[0]} channels")
print(f"Sampling rate:  {rec['fsamp']} Hz")
print(f"Duration:       {rec['emg'].shape[1] / rec['fsamp']:.1f} s")

## 2 – Explore the recording

The dataset is distributed as a **BIDS-flat** directory. Each recording consists of:

| File | Description |
|---|---|
| `*_emg.edf` | HD-EMG signal (EMG channels + Torque + JointAngle) |
| `*_channels.tsv` | Channel names, types, units, and `signal_electrode` pointer |
| `*_electrodes.tsv` | Physical electrode positions (x, y in mm) |
| `*_emg.json` | Sidecar: sampling frequency, task name |
| `*_events.tsv` | Behavioural phase events (onset, duration, event_type, description) |

`load_recording()` parses all five files and returns a dict:

| Key | Shape / Type | Description |
|---|---|---|
| `emg` | (n_channels, n_samples) | HD-EMG signal in mV |
| `effort` | (n_samples,) | Normalised effort (0–1, fraction of MVC) |
| `angle` | (n_samples,) | Joint angle in degrees |
| `fsamp` | int | Sampling frequency (2048 Hz) |
| `n_rows`, `n_cols` | int | Electrode grid dimensions |
| `channels` | DataFrame | Channel metadata from `*_channels.tsv` |
| `electrodes` | DataFrame | Electrode positions from `*_electrodes.tsv` |
| `events` | DataFrame | Behavioural phase events from `*_events.tsv` |
| `task_name` | str | Recording description |

### Dynamic task structure

The dynamic sub-dataset contains two contraction types:

| Contraction type | Task label | Description |
|---|---|---|
| Dynamic (no hold) | `dynamicNoHold` | Continuous force trajectory with no stationary plateaus — effort varies throughout the trial |
| Dynamic (with hold) | `dynamicWithHold` | Waypoint trajectory where the force is briefly held constant at each waypoint before transitioning to the next |

The recording loaded here belongs to the **with-hold** variant. Each stationary plateau is labelled `hold` in the events file, and the force-varying segments between plateaus are labelled `transition`.

In [ ]:
emg        = rec["emg"]       # (n_channels, n_samples)
effort     = rec["effort"]
angle      = rec["angle"]
fsamp      = rec["fsamp"]
n_rows, n_cols = rec["n_rows"], rec["n_cols"]
electrodes = rec["electrodes"]
n_channels, n_samples = emg.shape
duration_s = n_samples / fsamp
t = np.arange(n_samples) / fsamp

# Best channel by RMS (for time-series plot)
rms = np.sqrt(np.mean(emg ** 2, axis=1))
emg_ch = emg[int(np.argmax(rms))]
# bandpass-filter for display
b, a = butter(2, [10.0, 500.0], fs=fsamp, btype="band")
emg_ch = filtfilt(b, a, emg_ch)

### Signal overview

Let's plot the three main signals. The **effort profile** (% MVC) now follows a dynamic trajectory — force ramps between waypoints rather than holding flat — so the EMG amplitude will vary across the recording. The **joint angle** reflects the limb configuration at each phase, and the **best EMG channel** (highest RMS, bandpass-filtered 10–500 Hz) gives a quick read on signal quality throughout the trial.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, effort * 100)
axes[0].set_ylabel("Effort (% MVC)")
axes[0].set_title(f"{rec['task_name']}")

axes[1].plot(t, angle)
axes[1].set_ylabel("Angle (°)")

axes[2].plot(t, emg_ch, lw=0.5)
axes[2].set_ylabel("EMG (mV)")
axes[2].set_xlabel("Time (s)")

fig.tight_layout()
plt.show()

### Spatial activation map

Beyond the time domain it is useful to see **where on the electrode grid** the muscle is active. The `*_electrodes.tsv` sidecar provides the physical (x, y) position of every electrode in mm. By computing the RMS of each channel over the full trial and mapping it onto those positions we get a heatmap of surface activation — hot spots indicate electrodes that sit directly over the innervation zone or the region of highest motor unit density.

In [ ]:
# Build RMS heatmap using true electrode positions from _electrodes.tsv
emg_channels = rec["channels"][rec["channels"]["type"].str.upper() == "EMG"].copy()
elec_pos = electrodes.set_index("name")[["x", "y"]]
emg_channels = emg_channels.join(elec_pos, on="signal_electrode")
emg_channels["rms"] = np.sqrt(np.mean(emg ** 2, axis=1))

xs = np.sort(emg_channels["x"].unique())
ys = np.sort(emg_channels["y"].unique())
x_to_col = {x: i for i, x in enumerate(xs)}
y_to_row = {y: i for i, y in enumerate(ys)}

grid = np.full((len(ys), len(xs)), np.nan)
for _, ch in emg_channels.iterrows():
    grid[y_to_row[ch["y"]], x_to_col[ch["x"]]] = ch["rms"]

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(grid, aspect="auto", cmap="hot", origin="upper")

for _, ch in emg_channels.iterrows():
    row = y_to_row[ch["y"]]
    col = x_to_col[ch["x"]]
    ax.text(col, row, ch["name"], ha="center", va="center",
            fontsize=8, color="white", alpha=0.8)

ax.set_xlabel("Column")
ax.set_ylabel("Row")
ax.set_title("HD-EMG array – RMS per electrode")
fig.colorbar(im, ax=ax, label="RMS (mV)")
plt.tight_layout()
plt.show()

## 3 – Decompose with CBSS

CBSS (Convolutive Blind Source Separation) is a standard fastICA-based algorithm for
motor unit decomposition. It is the reference algorithm included with `muniverse`.

### What CBSS expects

* Input: **2D array** shaped `(n_channels, n_samples)`
* `load_recording()` already returns EMG in this layout — no reshaping needed.

In [ ]:
# EMG is already 2D from load_recording (n_channels, n_samples)
emg_2d = emg   # (n_channels, n_samples)
print(f"EMG ready for decomposition: {emg_2d.shape[0]} channels × {n_samples} samples")

### Choosing the analysis window

The `*_events.tsv` sidecar records every phase of the trial. For `dynamicWithHold` recordings, stationary **`hold`** events bracket each force plateau — these are the ideal windows for CBSS, which assumes stationary EMG statistics. For `dynamicNoHold` recordings there are no hold events, so the code falls back to the full active window between `muscle_on` and `muscle_off`.

The cell below reads the events table, picks the appropriate window, and passes it to the CBSS pre-processing config.

In [ ]:
# Load and adjust the CBSS configuration
cbss_cfg = json.loads(files("muniverse").joinpath("configs/cbss.json").read_text())

# Determine analysis window from events:
# Here we consider the data from muscle_on until muscle_off
events_df = rec["events"]
start_time = events_df.query("event_type == 'muscle_on'")['onset'].item()
end_time   = events_df.query("event_type == 'muscle_off'")['onset'].item()

cbss_cfg["preProcessingConfig"][2]["t_start"] = start_time
cbss_cfg["preProcessingConfig"][2]["t_end"]   = end_time
cbss_cfg["algorithmConfig"]["ica_iterations"] = 5
cbss_cfg["algorithmConfig"]["verbose"] = False

In [ ]:
# Run decomposition  (≈ 1–3 min depending on hardware)
print("Running CBSS decomposition …")

results, run_metadata = decompose_cbss(
    emg,
    fsamp,
    algorithm_config=cbss_cfg,
    meta={"filename": str(edf_path), "format": "edf"},
)

In [ ]:
spikes     = results["spikes"]        # DataFrame: onset, duration, sample, unit_id, event_type
silhouette = results["scores"]["sil"] # quality score per detected MU (0–1)

n_detected = results["sources"].shape[0]
mean_sil   = float(np.mean(silhouette)) if silhouette is not None and len(silhouette) > 0 else float("nan")
print(f"\nDetected {n_detected} motor units | mean silhouette = {mean_sil:.3f}")

### Decomposition strategy

The presence of **stationary hold windows** opens a natural two-phase decomposition approach:

1. **Phase 1 – Initialise on stationary segments.** Apply a standard algorithm (e.g. CBSS) to the hold windows, where the EMG statistics are approximately stationary. This yields stable, high-quality initial motor unit filters.

2. **Phase 2 – Track across the full dynamic signal.** Use the motor unit filters from Phase 1 as starting points for an adaptive decomposition algorithm that can track non-stationary activity across the force-varying transition segments between holds.

This strategy mirrors how decomposition is approached in experimental practice: leverage the simpler stationary portions to bootstrap decomposition, then extend tracking to the harder dynamic regime.

In this notebook we demonstrate **Phase 1**: CBSS decomposition of a hold window. The exported spike trains serve as a leaderboard submission and as starting material for Phase 2.

## 4 – Visualise decomposition results

In [ ]:
idx = 0

source = results["sources"][idx]  # (n_timestamps,)
t_src = np.arange(len(source)) / fsamp

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_src, source, lw=0.2)
ax.set_title(f"Source {idx} (silhouette = {silhouette[idx]:.3f})")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (a.u.)")
fig.tight_layout()
plt.show()

In [ ]:
if spikes.empty:
    print("No motor units detected – check your configuration or signal quality.")
else:
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(12, 6), gridspec_kw={"height_ratios": [1, 3]}, sharex=True
    )

    ax_top.plot(t, effort * 100, color="#2196F3", lw=1.5)
    ax_top.set_ylabel("Effort (% MVC)")
    ax_top.set_title(f"CBSS decomposition – {n_detected} MUs detected")

    for row_idx, (unit_id, group) in enumerate(spikes.groupby("unit_id")):
        sp_times = group["onset"].values
        ax_bot.vlines(sp_times, row_idx - 0.4, row_idx + 0.4, lw=0.5, color="#E53935", alpha=0.7)

    ax_bot.set_ylabel("Detected motor unit")
    ax_bot.set_xlabel("Time (s)")
    ax_bot.set_xlim(0, duration_s)
    ax_bot.set_ylim(-1, n_detected)

    fig.tight_layout()
    plt.show()

## 5 – Export for leaderboard upload

The MUnitQuest leaderboard expects a **BIDS-style tab-separated file** named after the source recording:

```
sub-syn001_task-dynamicWithHold_desc-decomposition_events.tsv
```

with the following columns:

| Column | Type | Description |
|---|---|---|
| `onset` | float | Spike time in **seconds** |
| `duration` | int | Always `0` (instantaneous event) |
| `sample` | int | Spike sample index |
| `unit_id` | int | Motor unit index (0-based) |
| `event_type` | str | Annotates `"motor-unit-spike"` |

One row per spike discharge. `export_events_file()` below handles the conversion and naming automatically.

In [ ]:
SUBMISSION_FOLDER = Path("../submissions/submission-dynamic")
SUBMISSION_FOLDER.mkdir(exist_ok=True)

In [ ]:
def export_events_file(spike_times, unit_ids, edf_path, fsamp, output_dir=None):
    """Write spike trains to a BIDS-style *_desc-decomposition_events.tsv.

    Parameters
    ----------
    spike_times : array-like of int
        Spike sample indices.
    unit_ids : array-like of int
        Motor unit label for each spike. Must be the same length as
        ``spike_times``.
    edf_path : str or Path
        Path to the source EDF file (used to derive the output filename).
    fsamp : int
        Sampling frequency in Hz.
    output_dir : str or Path, optional
        Directory to write the TSV. Defaults to the same directory as edf_path.

    Returns
    -------
    Path
        Absolute path of the written TSV file.
    """
    spike_times = np.asarray(spike_times)
    unit_ids    = np.asarray(unit_ids)
    assert len(spike_times) == len(unit_ids), (
        f"spike_times and unit_ids must have the same length "
        f"(got {len(spike_times)} and {len(unit_ids)})"
    )

    df = pd.DataFrame({
        "onset":       np.round(spike_times / fsamp, 6),
        "duration":    0,
        "sample":      spike_times.astype(int),
        "unit_id":     unit_ids.astype(int),
        "event_type": "motor-unit-spike",
    })
    df = df.sort_values("onset").reset_index(drop=True)

    edf_path = Path(edf_path)
    stem = edf_path.stem.replace("_emg", "_desc-decomposition")
    out_dir = Path(output_dir) if output_dir else edf_path.parent
    out_path = out_dir / f"{stem}_events.tsv"
    df.to_csv(out_path, sep="\t", index=False, na_rep="n/a")
    return out_path

In [ ]:
out_path = export_events_file(spikes["sample"].values, spikes["unit_id"].values, edf_path, fsamp,
                              output_dir=SUBMISSION_FOLDER)
print(f"Saved decomposition outputs to {out_path}")

## 6 – Prepare metadata for your submission

Every submission to MUnitQuest must include a metadata file alongside each decomposition events TSV, e.g.,

`recording-XXX_challenge-dynamic_desc-decomposition_log.json`

At minimum it must contain:

| Field | Type | Required sub-keys | Description |
|---|---|---|---|
| `GeneratedBy` | `list[dict]` | `Name`, `Description`, `CodeURL`, `License`| Algorithm name, a short plain-English description, and a URL pointing to the code (GitHub, GitLab, or similar). `Version` is optional. **`CodeURL` is required even if your repository is private** — it is used to verify algorithm identity, not to access the code. |
| `Execution` | `dict` | `Runtime` | Total wall-clock execution time for the decomposition (seconds, as a number) |
| `Environment`| `dict` | — | Hardware used: CPU model, total RAM, and GPU (if applicable) |

This metadata helps contextualise performance differences across submissions.

### Tracking execution time

Wrap your decomposition call with `time.time()` to measure wall-clock duration:

```python
import time
t0 = time.time()
results = my_decomposition_algorithm(...)
runtime_seconds = time.time() - t0
```

If you use `muniverse`'s `decompose_cbss`, timing is already captured in `run_metadata['Execution']['Timing']` — no manual instrumentation needed (see below).

### Recording hardware information

The cells below use `platform`, `subprocess`, and `psutil` to collect CPU, RAM, and GPU details automatically.

In [ ]:
import os, platform, subprocess

cpu_info = {
    "model":   platform.processor() or platform.machine() or "unknown",
    "n_cores": os.cpu_count(),
    "system":  platform.system(),
    "node":    platform.node(),
}

try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / 1e9)
except ImportError:
    ram_gb = "unknown"

try:
    gpu = subprocess.check_output(
        "nvidia-smi --query-gpu=name --format=csv,noheader", shell=True
    ).decode().strip()
except Exception:
    gpu = "none"

Now assemble the full metadata dict. The manual approach below works regardless of which decomposition library you use — just substitute `runtime_seconds` with however you measured it:

In [ ]:
from datetime import datetime

t_start = datetime.fromisoformat(run_metadata['Execution']['Timing']['Start'])
t_end   = datetime.fromisoformat(run_metadata['Execution']['Timing']['End'])

runtime_seconds = (t_end - t_start).total_seconds()

Next, add a small description to the metadata_dict

In [ ]:
description: str = "Tutorial notebook to decompose a recording from the MUnitQuest Algorithm Challenge - Dynamic Task."

run_metadata["GeneratedBy"][0]["Description"] = description

In [ ]:
metadata_log = {
    "GeneratedBy": [
        {
            "Name": "MUnitQuest Tutorials",
            "Description": description,
            "CodeURL": "https://github.com/MUnitQuest/MUnitQuest_tutorials/tree/main/notebooks/02_familiarisation_dynamic.ipynb",
            "Version": "n/a",
            "Container": "n/a"
        }
    ],
    "Execution": {
        "Runtime": runtime_seconds
    },
    "RuntimeEnvironment": {
        "CPU": cpu_info,
        "GPU": gpu,
        "RAM": ram_gb
    }
}

If you decompose with `muniverse`'s `decompose_cbss`, the `run_metadata` dict already contains all of the above in a structured form — hardware, timing, and algorithm provenance are captured automatically, so you can build the log in one line:

In [ ]:
metadata_log = {
    "GeneratedBy": run_metadata['GeneratedBy'],
    "Execution": run_metadata['Execution'],
    "RuntimeEnvironment": run_metadata['RuntimeEnvironment']
}

In [ ]:
def export_metadata(metadata_log, edf_path, output_dir=None):
    """Write a metadata log dict to a *_desc-metadata_log.json file.

    Parameters
    ----------
    metadata_log : dict
        Metadata to serialise (GeneratedBy, Runtime, Environment, …).
    edf_path : str or Path
        Path to the source EDF file (used to derive the output filename).
    output_dir : str or Path, optional
        Directory to write the JSON. Defaults to the same directory as edf_path.

    Returns
    -------
    Path
        Absolute path of the written JSON file.
    """
    edf_path = Path(edf_path)
    stem = edf_path.stem.replace("_emg", "_desc-decomposition_log")
    out_dir = Path(output_dir) if output_dir else edf_path.parent
    out_path = out_dir / f"{stem}.json"
    out_path.write_text(json.dumps(metadata_log, indent=2))
    return out_path

In [ ]:
out_path = export_metadata(metadata_log, edf_path, output_dir=SUBMISSION_FOLDER)
print(f"Saved metadata to {out_path}")

## 7 – Decompose the full dataset and submit

You have now decomposed a single dynamic recording end-to-end. The familiarisation set contains 100 recordings across two contraction types (`dynamicWithHold` and `dynamicNoHold`) — repeat the steps above for each one, but with your algorithm! 

### Submission format

For each recording, `export_events_file` and `export_metadata` write a pair of files into your results directory:

| File | Content |
|---|---|
| `*_desc-decomposition_events.tsv` | Spike trains (one row per discharge) |
| `*_desc-decomposition_log.json` | Algorithm provenance, runtime, and hardware |

A few things to keep in mind as you scale up:

- The `*_events.tsv` sidecar is present for every recording and always contains `muscle_on` / `muscle_off` events — any spikes outside these events will be disregarded during evaluation.
- For `dynamicWithHold` recordings, the hold windows identified in the events file are the best windows for CBSS initialisation. For `dynamicNoHold` recordings the code falls back to the full active window between `muscle_on` and `muscle_off`.
- Both export functions derive the output filename automatically from the EDF path, so you can loop over recordings without name collisions.

### Validate your submission

Run `validate_submission()` on your results directory to catch formatting errors before upload:

In [ ]:
from utils import validate_submission

is_valid, errors, warnings = validate_submission(
    submission_dir=SUBMISSION_FOLDER,
    data_dir=DATASET_ROOT,
)

Fix any reported errors before proceeding — warnings (e.g. skipped recordings) are non-fatal. And finally, create a `.zip` archive to submit. Assuming you collected all your decomposition results into `submission-dynamic/`, simply do

```bash
zip -j my_submission.zip submission-dynamic/*
```

Then head to the [MUnitQuest Codabench submission page](https://www.codabench.org/competitions/15749/) to compete!